# Prepare MISR data

This notebook uses the MISR package to convert MISR hdf files into an xarray format. Not every piece of information contained in the hdf file is transferred to the xarray. Only the pieces relavant to AT3D.

In [ ]:
# conda env 'misr'
import MisrToolkit as mtk
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.MISR import hdf2xr, grp2xr

In [4]:
## Set data path
misr_data_dir = PROJECT_ROOT / 'data' / 'raw' / 'MISR'

## Specify MISR parameters
path = 117
orbit = 105254
cameras = ['DF', 'CF', 'BF', 'AF', 'AN', 'AA', 'BA', 'CA', 'DA']

## Define geographic region
center_lat = 19.266267063099104
center_lon = 119.72428415176724
region = mtk.MtkRegion(center_lat, center_lon, 8.8*3, 8.8*3, "km")

## Load MISR red band images
imagery = grp2xr(misr_data_dir, path, orbit, cameras, region)
imagery.to_netcdf(PROJECT_ROOT / 'data' / 'interim' / f'MISR_RAD_O{orbit:06}_{imagery.regionID}.nc')

## Load geometric parameters
gmp_file = misr_data_dir / f'MISR_AM1_GP_GMP_P{path}_O{orbit}_F03_0013.hdf'
field_list_solar = ['SolarZenith', 'SolarAzimuth']
field_list_vza = [f'{c.lower().capitalize()}Zenith' for c in cameras]
field_list_vaa = [f'{c.lower().capitalize()}Azimuth' for c in cameras]
field_list = field_list_solar + field_list_vza + field_list_vaa
gmp = hdf2xr(gmp_file, 'GeometricParameters', field_list, region)
gmp.to_netcdf(PROJECT_ROOT / 'data' / 'interim' / f'MISR_GEO_O{orbit:06}_{gmp.regionID}.nc')

## Load CTH
tc_cloud_file = misr_data_dir / f'MISR_AM1_TC_CLOUD_P{path:03d}_O{orbit:06d}_F01_0001.hdf'
cth = hdf2xr(tc_cloud_file, 'Stereo_1.1_km', ['CloudTopHeight', 'StereoDerivedCloudMask'], region)
cth.to_netcdf(PROJECT_ROOT / 'data' / 'interim' / f'MISR_CTH_O{orbit:06}_{cth.regionID}.nc')

## Load CMV
cmv = hdf2xr(tc_cloud_file, 'Motion_17.6_km', ['CloudTopHeightOfMotion', 'CloudMotionNorthward', 'CloudMotionEastward'], region)
cmv.to_netcdf(PROJECT_ROOT / 'data' / 'interim' / f'MISR_CMV_O{orbit:06}_{cmv.regionID}.nc')
